# O'FAIRe scores of the EarthPortal ontologies

This notebook turns the O'FAIRe results into one table: **one row per ontology**, with its overall
scores and the points it earned on each of the 61 questions.

- **I.** Loading the snapshot
- **II.** Building the table
- **III.** Checking the table
- **IV.** Saving the table

In [1]:
# Imports
# Standard library imports
import json
import math

# third-party 
import pandas as pd

# local
from config import *

# I. Loading the snapshot

In [2]:
# We load the catalogue (names, kinds) and the O'FAIRe results saved by notebook 01
catalogue = {o["acronym"]: o for o in json.load(open(RAW / "catalogue.json", encoding="utf-8"))}
ofaire = json.load(open(RAW / "ofaire.json", encoding="utf-8"))
print(f"Snapshot {RUN_DATE}: {len(catalogue)} entries, {len(ofaire)} O'FAIRe results")

Snapshot 2026-10-08: 92 entries, 92 O'FAIRe results


In [3]:
# A quick inspection of catalogue
df_cat = pd.DataFrame(catalogue.values())
df_cat.head()

,id,acronym,administeredBy,viewingRestriction,acl,projects,views,name,group,hasDomain,notes,reviews,viewOf,flat,ontologyType,summaryOnly,sampleQueries,doNotUpdate,@id,@type
0,https://data.earthportal.eu/ontologies/CALATHE,CALATHE,[https://data.earthportal.eu/users/vcagdas],public,[],[],[],Cadastre and Land Administration Thesaurus,[https://data.earthportal.eu/groups/W3C],"[https://data.earthportal.eu/categories/EARTH,...",[],[],None,None,https://data.earthportal.eu/ontology_types/ONT...,False,[],None,https://data.earthportal.eu/ontologies/CALATHE,http://data.bioontology.org/metadata/Ontology
1,https://data.earthportal.eu/ontologies/ENVTHES,ENVTHES,[https://data.earthportal.eu/users/ecoportal],public,[],[],[],Environmental Thesaurus,[],[https://data.earthportal.eu/categories/ENV],[https://data.earthportal.eu/notes/17ee4350-c6...,[],None,None,https://data.earthportal.eu/ontology_types/ONT...,False,[],None,https://data.earthportal.eu/ontologies/ENVTHES,http://data.bioontology.org/metadata/Ontology
2,https://data.earthportal.eu/ontologies/N4E,N4E,[https://data.earthportal.eu/users/aurioldegbe...,public,[],[],[],NFDI4Earth Ontology,[https://data.earthportal.eu/groups/NFDI],[https://data.earthportal.eu/categories/ENV],[],[],None,None,https://data.earthportal.eu/ontology_types/ONT...,False,[],None,https://data.earthportal.eu/ontologies/N4E,http://data.bioontology.org/metadata/Ontology
3,https://data.earthportal.eu/ontologies/NVS_C17,NVS_C17,[https://data.earthportal.eu/users/alexk],public,[],[],[],ICES Platform Codes,[https://data.earthportal.eu/groups/BODC],"[https://data.earthportal.eu/categories/EARTH,...",[],[],None,None,https://data.earthportal.eu/ontology_types/ONT...,False,[],None,https://data.earthportal.eu/ontologies/NVS_C17,http://data.bioontology.org/metadata/Ontology
4,https://data.earthportal.eu/ontologies/NVS_C19,NVS_C19,[https://data.earthportal.eu/users/alexk],public,[],[],[],SeaVoX salt and fresh water body gazetteer,[https://data.earthportal.eu/groups/BODC],"[https://data.earthportal.eu/categories/EARTH,...",[],[],None,None,https://data.earthportal.eu/ontology_types/ONT...,False,[],None,https://data.earthportal.eu/ontologies/NVS_C19,http://data.bioontology.org/metadata/Ontology


In [4]:
# Same for O'FAIRe results
df_ofaire = pd.DataFrame(ofaire).T
df_ofaire.head()

,Findable,Accessible,Interoperable,Reusable,score,normalizedScore,maxCredits,executionTime
CALATHE,{'F1': {'label': 'Ontologies and ontology meta...,{'A1': {'label': 'Ontologies and ontology meta...,{'I1': {'label': 'Ontologies and ontology meta...,{'R1': {'label': 'Ontologies and ontology meta...,245.0,51,478.0,615
ENVTHES,{'F1': {'label': 'Ontologies and ontology meta...,{'A1': {'label': 'Ontologies and ontology meta...,{'I1': {'label': 'Ontologies and ontology meta...,{'R1': {'label': 'Ontologies and ontology meta...,273.0,57,478.0,642
N4E,{'F1': {'label': 'Ontologies and ontology meta...,{'A1': {'label': 'Ontologies and ontology meta...,{'I1': {'label': 'Ontologies and ontology meta...,{'R1': {'label': 'Ontologies and ontology meta...,254.0,53,478.0,920
NVS_C17,{'F1': {'label': 'Ontologies and ontology meta...,{'A1': {'label': 'Ontologies and ontology meta...,{'I1': {'label': 'Ontologies and ontology meta...,{'R1': {'label': 'Ontologies and ontology meta...,269.75,56,478.0,19022
NVS_C19,{'F1': {'label': 'Ontologies and ontology meta...,{'A1': {'label': 'Ontologies and ontology meta...,{'I1': {'label': 'Ontologies and ontology meta...,{'R1': {'label': 'Ontologies and ontology meta...,264.75,55,478.0,3442


In [5]:
# We check our hypothesis about some questions:
# We flatten the O'FAIRe results into a long table: one row per (ontology, question)
records = []
for acronym, result in ofaire.items():
    for principle in ["Findable", "Accessible", "Interoperable", "Reusable"]:
        for sub_principle, block in result[principle].items():
            if not isinstance(block, dict) or "results" not in block:
                continue                       # skip the principle's own score fields
            for qid, q in block["results"].items():
                records.append({"acronym": acronym, "kind": kind_of(catalogue[acronym]),
                                "sub_principle": sub_principle, "question": qid,
                                "score": q["score"], "max_credits": q["maxCredits"],
                                "explanation": q.get("explanation") or ""})
answers = pd.DataFrame(records)
pd.set_option("display.max_colwidth", None)        # show the full messages
print(f"{len(answers)} answers = {answers.acronym.nunique()} ontologies x {answers.question.nunique()} questions")

5612 answers = 92 ontologies x 61 questions


In [6]:
answers.head()

,acronym,kind,sub_principle,question,score,max_credits,explanation
0,CALATHE,artefact,F1,F1Q1,9.0,9.0,Ontology URI is present and valid.
1,CALATHE,artefact,F1,F1Q2,0.0,11.0,External identifier is absent.
2,CALATHE,artefact,F1,F1Q3,12.0,12.0,Ontology metadata clearly identified: the repository explicitly identifies its metadata.
3,CALATHE,artefact,F1,F1Q4,0.0,9.0,Version URI is absent.
4,CALATHE,artefact,F2,F2Q1,6.0,16.0,The ontology is described with 3 'MIRO must' properties.


In [7]:
# Claim 1: ten questions always answer 0 with "cannot be evaluated", whatever the ontology
not_implemented = answers[answers.explanation.str.contains("cannot be evaluated")]
check1 = not_implemented.groupby("question").agg(
    ontologies=("acronym", "nunique"),
    distinct_scores=("score", lambda s: sorted(s.unique().tolist())),
    max_credits=("max_credits", "first"),
    messages=("explanation", lambda s: sorted(s.unique().tolist())))
print(f"{len(check1)} questions not implemented, worth {check1.max_credits.sum()} points in total")
check1

10 questions not implemented, worth 70.0 points in total


,ontologies,distinct_scores,max_credits,messages
question,,,,
F4Q3,92,[0.0],8.0,[This question cannot be evaluated in the repository at this time.]
I2Q3,92,[0.0],3.0,[This question cannot be evaluated in the repository at this time.]
I2Q5,92,[0.0],7.0,[This question cannot be evaluated in the repository at this time.]
I3Q1,92,[0.0],20.0,[This question cannot be evaluated in the repository at this time.]
I3Q2,92,[0.0],6.0,[This question cannot be evaluated in the repository at this time.]
R1.2Q5,92,[0.0],2.0,[This question cannot be evaluated in the repository at this time.]
R1Q3,92,[0.0],7.0,[This question cannot be evaluated in the repository at this time.]
R1Q4,92,[0.0],6.0,[This question cannot be evaluated in the repository at this time.]
R1Q5,92,[0.0],6.0,[This question cannot be evaluated in the repository at this time.]


In [8]:
# Claim 2: F3Q1 is always 0, F3Q2 and F3Q3 are always full, R1.3Q2 is always 0
special = ["F3Q1", "F3Q2", "F3Q3", "R1.3Q2"]
check2 = answers[answers.question.isin(special)].groupby("question").agg(
    ontologies=("acronym", "nunique"),
    max_credits=("max_credits", "first"),
    distinct_scores=("score", lambda s: s.value_counts().to_dict()),
    messages=("explanation", lambda s: sorted(s.unique().tolist())))
check2

,ontologies,max_credits,distinct_scores,messages
question,,,,
F3Q1,92,21.0,{0.0: 92},[This question ]
F3Q2,92,11.0,{11.0: 92},[The ontology repository provides ontology metadata in an external file.]
F3Q3,92,10.0,{10.0: 92},[Bidirectional links between the ontology and its metadata.]
R1.3Q2,92,20.0,{0.0: 92},[The ontology does not provide information about its community set or group.]


In [9]:
# The F3 block counts 21 points (= F3Q2 + F3Q3), although F3Q1 alone also claims 21
f3_block = pd.Series({a: (r["Findable"]["F3"]["score"], r["Findable"]["F3"]["maxCredits"]) for a, r in ofaire.items()})
print("F3 block (score, maximum):", f3_block.value_counts().to_dict())

# This is why the 61 question maxima add up to more than 478
question_max = answers.groupby("question").max_credits.first()
print(f"Sum of the 61 question maxima: {question_max.sum()}; "
      f"478 + {question_max['F3Q1']} (F3Q1) - 2 (I1) = {478 + question_max['F3Q1'] - 2}")

# R1.3Q2 rewards only groups named OBO-FOUNDRY, WHEAT, CROP or INRAE: does EarthPortal have one?
groups = sorted({g.rsplit("/", 1)[-1] for o in catalogue.values() for g in (o.get("group") or [])})
print("EarthPortal groups:", groups)
print("Groups R1.3Q2 would reward:",
      [g for g in groups if any(w in g.upper() for w in ["OBO-FOUNDRY", "WHEAT", "CROP", "INRAE"])] or "none")

F3 block (score, maximum): {(21.0, 21.0): 92}
Sum of the 61 question maxima: 497.0; 478 + 21.0 (F3Q1) - 2 (I1) = 497.0
EarthPortal groups: ['ACTRIS', 'AERIS', 'BODC', 'BRGM', 'DATA_TERRA', 'EPOS', 'ESIP', 'FORMATER', 'IFREMER', 'NFDI', 'ODATIS', 'OGC', 'SEXTANT', 'THEIA', 'W3C']
Groups R1.3Q2 would reward: none


# II. Building the table

Two details matter here:

- Ten questions are **not implemented** in O'FAIRe: its code answers 0 with the message *"This question
  cannot be evaluated in the repository at this time"*, whatever the ontology and whatever the portal.
  That 0 does not mean the ontology failed, so we store those answers as empty (`NaN`).
- Two questions behave in a special way (see `reference/ofaire_known_issues.csv`): F3Q1 is always 0
  while F3Q2 and F3Q3 are given in full to every ontology, and R1.3Q2 only rewards AgroPortal groups or
  the OBO library. We keep the second one but count it as unreachable.

From this we compute a **reachable maximum**: the sum of the maximum of each question an EarthPortal
ontology can score on. It is 386 points: 478, minus 70 (the ten questions not implemented), minus 20
(R1.3Q2), minus 2 (the five questions of I1 add up to 42 points, but O'FAIRe counts 44 for I1).

In [10]:
known_issues = pd.read_csv(KNOWN_ISSUES)
known_issues.head()

,question,treatment,reason,evidence
0,F3Q1,not_evaluated,"The code gives 0 to every artefact, with the unfinished explanation ""This question"". F3Q2 and F3Q3 are given in full to every artefact, and the F3 maximum (21 = 11 + 10) counts only these two, so F3Q1 is never a real loss. The 478 total already counts F3 once, so the reachable maximum is not reduced.","F3.java line 25 (agroportal/fairness, commit 24def6b); O'FAIRe results of 2026-09-28 (91/91 entries)"
1,R1.3Q2,unreachable,"The code gives credits only to groups whose name contains OBO-FOUNDRY, WHEAT, CROP or INRAE. No EarthPortal group has such a name, so an EarthPortal artefact cannot earn them. Removed from the reachable maximum.","R13.java lines 50 to 63 (agroportal/fairness, commit 24def6b); O'FAIRe results of 2026-09-28 (0 points for 91/91 entries)"


In [11]:
# Info on known issues
known_issues.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   question   2 non-null      object
 1   treatment  2 non-null      object
 2   reason     2 non-null      object
 3   evidence   2 non-null      object
dtypes: object(4)
memory usage: 196.0+ bytes


In [12]:
known = dict(zip(known_issues.question, known_issues.treatment))
known

{'F3Q1': 'not_evaluated', 'R1.3Q2': 'unreachable'}

In [13]:
PRINCIPLES = ["Findable", "Accessible", "Interoperable", "Reusable"]
rows, questions = [], {}

for acronym, result in ofaire.items():
    row = {"acronym": acronym, "name": catalogue[acronym]["name"], "kind": kind_of(catalogue[acronym]),
           "normalized_score": result["normalizedScore"], "total_points": result["score"]}
    for principle in PRINCIPLES:
        for sub_principle, block in result[principle].items():
            if not isinstance(block, dict) or "results" not in block:
                continue                       # skip the principle's own score fields
            for qid, q in block["results"].items():
                not_implemented = "cannot be evaluated" in (q.get("explanation") or "")
                evaluable = not not_implemented and known.get(qid) != "not_evaluated"
                row[qid] = q["score"] if evaluable else math.nan
                questions.setdefault(qid, {"question": qid, "principle": principle,
                                           "sub_principle": sub_principle, "max_credits": q["maxCredits"],
                                           "evaluable": evaluable,
                                           "reachable": not not_implemented and known.get(qid) != "unreachable",
                                           "text": q["question"].strip()})
    rows.append(row)
questions = pd.DataFrame(questions.values())

# The reachable maximum: the sum of the maximum of each question an EarthPortal ontology can score on
reachable_max = int(questions[questions.evaluable & questions.reachable].max_credits.sum())
for row in rows:
    row["reachable_max"] = reachable_max
    row["reachable_score_pct"] = round(100 * row["total_points"] / reachable_max, 1)

# We put the overall columns first, then the 61 questions in O'FAIRe's own order
front = ["acronym", "name", "kind", "normalized_score", "total_points", "reachable_max", "reachable_score_pct"]
scores = pd.DataFrame(rows)[front + list(questions.question)]
scores = scores.sort_values(["kind", "reachable_score_pct"], ascending=[True, False]).reset_index(drop=True)
scores.head()

,acronym,name,kind,normalized_score,total_points,reachable_max,reachable_score_pct,F1Q1,F1Q2,F1Q3,...,R1.2Q2,R1.2Q3,R1.2Q4,R1.2Q5,R1.2Q6,R1.2Q7,R1.2Q8,R1.3Q1,R1.3Q2,R1.3Q3
0,OSO,Observatories of the Seas Ontology,artefact,76,364.36,386,94.4,9.0,11.0,12.0,...,4.0,6.0,4.0,NaN,6.0,4.0,2.0,10.0,0.0,6.0
1,NVS_OD1,ODATIS aggregation parameters and Essential Variable names,artefact,61,292.75,386,75.8,9.0,0.0,12.0,...,2.0,2.0,4.0,NaN,2.0,2.0,2.0,10.0,0.0,6.0
2,EASY_DATA_VOCAB,EaSy Data Unified Thesaurus,artefact,60,288.00,386,74.6,9.0,0.0,12.0,...,4.0,4.0,4.0,NaN,4.0,2.0,2.0,10.0,0.0,6.0
3,NVS_P06,Governance for vocabularies used within the data centre,artefact,60,287.75,386,74.5,9.0,0.0,12.0,...,0.0,2.0,4.0,NaN,2.0,2.0,0.0,5.0,0.0,6.0
4,NVS_P07,Climate and Forecast Standard Names,artefact,60,287.75,386,74.5,9.0,0.0,12.0,...,2.0,2.0,4.0,NaN,6.0,2.0,0.0,5.0,0.0,6.0


# III. Checking the table

In [14]:
# We check that the question scores add up to O'FAIRe's total, for every ontology
question_columns = list(questions.question)
if (scores[question_columns].sum(axis=1) - scores.total_points).abs().max() < 1e-6:
    print("For every ontology, the question scores add up exactly to O'FAIRe's total.")
else:
    print("Warning: for some ontologies, the question scores do not add up to O'FAIRe's total.")

# We check how O'FAIRe normalises: total / 478 * 100, rounded down
if (scores.normalized_score == (100 * scores.total_points / 478).apply(math.floor)).all():
    print("O'FAIRe's normalised score is the total out of 478, rounded down.")

# We check the reachable maximum against the criteria: 478, minus the questions without a usable
# answer or unreachable, minus the 2 points of I1 that no question gives
excluded = questions[~(questions.evaluable & questions.reachable) & (questions.question != "F3Q1")].max_credits.sum()
print(f"Questions an EarthPortal ontology can score on: {(questions.evaluable & questions.reachable).sum()} of {len(questions)}; "
      f"reachable maximum: {reachable_max} points = 478 - {excluded} - 2")
scores.kind.value_counts()

For every ontology, the question scores add up exactly to O'FAIRe's total.
O'FAIRe's normalised score is the total out of 478, rounded down.
Questions an EarthPortal ontology can score on: 49 of 61; reachable maximum: 386 points = 478 - 90.0 - 2


kind
artefact         68
view             19
metadata-only     5
Name: count, dtype: int64

In [15]:
# The views, with the artefact each one is a view of
views = scores.loc[scores.kind == "view", ["acronym", "name", "reachable_score_pct"]].copy()
views["view_of"] = views.acronym.map(lambda a: catalogue[a]["viewOf"].rsplit("/", 1)[-1])
print(f"{len(views)} views")
views.sort_values(["view_of", "acronym"]).reset_index(drop=True)

19 views


,acronym,name,reachable_score_pct,view_of
0,AER_INS,AERIS Instruments,57.0,AERIS_THES
1,AER_PFM,AERIS Platforms,57.0,AERIS_THES
2,AER_PJT,AERIS Projects,55.7,AERIS_THES
3,AER_PRM,AERIS Parameters,57.0,AERIS_THES
4,EASYDATA_KW,Vocabulaire des mots clefs,64.5,EASY_DATA_VOCAB
5,EASYDATA_THEMES,Vocabulaire thématique,58.3,EASY_DATA_VOCAB
6,FMT_DT,FormaTerre Disciplines and Themes,52.8,FORMATER_VOCAB
7,FMT_FOI,FormaTerre Feature of Interest,53.9,FORMATER_VOCAB
8,FMT_INT,FormaTerre Instruments,52.8,FORMATER_VOCAB
9,FMT_NET,FormaTerre Networks,53.9,FORMATER_VOCAB


**Conclusion**: the table is consistent with O'FAIRe. Artefacts, views and metadata-only records are
kept apart in the `kind` column; the analysis focuses on the artefacts.

In [16]:
# The metadata-only records (no content uploaded)
metadata_only = scores.loc[scores.kind == "metadata-only", ["acronym", "name", "normalized_score", "reachable_score_pct"]]
print(f"{len(metadata_only)} metadata-only records")
metadata_only.reset_index(drop=True)

5 metadata-only records


,acronym,name,normalized_score,reachable_score_pct
0,TEST,Test semantic,43,54.4
1,HG,Hello Go,43,53.9
2,SW,Semantic Web,43,53.9
3,HIG,HI Go,43,53.9
4,GTD,GO TO DO,43,53.9


# IV. Saving the table

In [17]:
scores.to_csv(DATA / f"ofaire_scores_{RUN_DATE}.csv", index=False)
questions.to_csv(DATA / f"ofaire_questions_{RUN_DATE}.csv", index=False)
print(f"Saved the two tables in {DATA.relative_to(REPO)}/")

Saved the two tables in earthportal_sa_audit/data/
